<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Flujo_personas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalamos ultralytics
#ModelosYOLO de Ultralytics
#una arquitectura de IA de última generación
#diseñada para el modelado de IA de visión de alta precisión.
!pip install ultralytics

In [ ]:
#opencv-python-headless es el paquete de OpenCV para Python
#sin funcionalidades de interfaz gráfica (GUI)
#diseñado para su uso en servidores, entornos de nube o Docker,
#donde no se requiere visualización de ventanas
!pip install opencv-python-headless

In [ ]:
#Descargamos el video de las personas caminando
!wget -O personas_caminando.mp4 https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi

--2025-12-17 14:45:30--  https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8131690 (7.8M) [video/msvideo]
Saving to: ‘personas_caminando.mp4’

personas_caminando. 100%[===================>]   7.75M  --.-KB/s    in 0.05s   

2025-12-17 14:45:31 (153 MB/s) - ‘personas_caminando.mp4’ saved [8131690/8131690]



In [ ]:
#Importamos las librerías necesarias
from ultralytics import YOLO
import cv2
import numpy as np

In [ ]:
#Cargamos el modelo yolov8n
#YOLOv8 el modelo de detección de objetos de
#última generación para visión por computadora
model = YOLO("yolov8n.pt")

In [ ]:
#cargamos el video para procesarlo
cap = cv2.VideoCapture("personas_caminando.mp4")
assert cap.isOpened(), "No se pudo abrir el video"

In [ ]:
#Se obtienen los fps, w y h
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [ ]:
fps, w, h

(10.0, 768, 576)

In [ ]:
#Establecemos el nombre de salida del video
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter("personas_caminandot.avi",
                      fourcc, fps, (w, h))
assert out.isOpened(), "VideoWriter falló"

In [ ]:
#Realizamos la detección de personas y el flujo
history = []
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    results = model(frame, verbose=False)
    count = 0

  #Se revisa si existen personas en el frame
  #y se realiza el conteo
    for r in results:
        for box in r.boxes:
            if int(box.cls[0]) == 0:
                count += 1

    history.append(count)
    avg = int(np.mean(history[-30:]))

    # Umbrales de flujo peatonal
    # Número de personas en el frame
    if avg < 3:
        label, color = "FLUJO PEATONAL BAJO", (0,255,0)
    elif avg < 7:
        label, color = "FLUJO PEATONAL MEDIO", (0,255,255)
    else:
        label, color = "FLUJO PEATONAL ALTO", (0,0,255)

    # Se genera un rectangulo y se coloca el número de personas
    #y el estado del flujo
    cv2.rectangle(frame, (0,0), (w,95), color, -1)
    cv2.putText(frame, f"Personas detectadas: {count}", (20,35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
    cv2.putText(frame, f"Estado: {label}", (20,70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)

    out.write(frame)

cap.release()
out.release()

print("Video de flujo peatonal generado")

Video de flujo peatonal generado
